# So sánh TA-SQL · V-SQL (tái hiện paper) · V-SQL+ (cải tiến) trên BIRD mini-dev — Colab T4

Notebook chạy **cùng một bộ dữ liệu như paper** (BIRD mini-dev, 500 câu, 11 CSDL, tính đủ 500 dòng) với ba model mã nguồn mở
qua Ollama trên GPU T4: `qwen3.5:0.8b`, `qwen3:8b`, `qwen3.5:9b`.

| Cấu hình | Nội dung |
|---|---|
| `tasql` | **TA-SQL** (Qu et al., ACL Findings 2024) — baseline mà paper V-SQL so sánh. Cài lại theo mã nguồn chính thức: TASL (dummy SQL → trích bảng/cột) + TALOG (biểu diễn ký hiệu SR → SQL), prompt chép nguyên văn, mô tả cột `column_meaning.json` của TA-SQL |
| `vsql` | **V-SQL giữ đúng theo paper**: view theo các phép JOIN của SQL đáp án (mục 6), kiểu Hình 3; schema trình bày như Bảng 6; giai đoạn 1 prompt Hình 4, giai đoạn 2 LLM khôi phục về bảng gốc bằng prompt Hình 5 (nguyên văn) |
| `vsqlplus` | **V-SQL+ (cải tiến)**, giữ hai giai đoạn của paper và kết hợp 2 hướng: **(1) tự động sinh view** từ khóa ngoại (chỉ nối N-1, kiểm số dòng, tên cột sạch) và **(2) chọn view đúng bằng truy hồi lai** BM25 + embedding, bổ trợ bởi truy hồi mức cột gộp lên view, liên kết giá trị thật trong DB, bầu chọn nhiều mẫu, schema lai (quay về bảng gốc khi LLM dùng cột không có trong view), CSDL nhỏ thì đưa hết view |

Biến thể đối chiếu (ABLATION): `vsqlplus_all` = chỉ hướng 1; `vsqlplus_novote` / `_novl` / `_nofb` = bỏ từng thành phần của hướng 2;
`vsqlplus_det` = giải mã tất định thay giai đoạn 2 (ngoài đề xuất, không tốn thêm lượt gọi).

**Hướng dẫn chi tiết từng bước, kết quả mong đợi và cách xử lý lỗi:** `docs/HUONG_DAN_COLAB_SO_SANH.md` trong repo.

**Cách dùng:** `Runtime → Change runtime type → T4 GPU`, rồi chạy lần lượt từng ô. Kết quả lưu vào Google Drive
(`MyDrive/vsqlplus_ketqua/<model>/`); nếu Colab ngắt, chạy lại từ Bước 1 — phần đã xong được khôi phục và bỏ qua.

**Thời gian ước tính trên T4** (≈ 4.500 lượt gọi LLM / model cho 3 phương pháp): `qwen3.5:0.8b` ~1–2 giờ, `qwen3:8b` và
`qwen3.5:9b` mỗi model ~6–10 giờ. Nên chạy **mỗi model một phiên** (chọn ở Bước 1); ô chạy chính in thời gian còn lại.

In [ ]:
#@title Bước 1. Cấu hình
#@markdown Chọn model chạy trong phiên này (chạy lần lượt). Với Colab miễn phí nên chọn 1 model mỗi phiên.
CHAY_qwen35_08b = True  #@param {type:"boolean"}
CHAY_qwen3_8b = False  #@param {type:"boolean"}
CHAY_qwen35_9b = False  #@param {type:"boolean"}
#@markdown ---
#@markdown **Số câu**: 0 = đủ 500 câu như paper; số khác = mẫu phân tầng theo CSDL × độ khó (vd 50 để chạy thử).
SAMPLE = 0  #@param {type:"integer"}
#@markdown Chạy thêm các biến thể bỏ từng thành phần của V-SQL+ (ablation, tốn thêm ~40% thời gian).
ABLATION = False  #@param {type:"boolean"}
#@markdown Mô hình embedding cho truy hồi lai (Ollama). Để trống = chỉ BM25.
EMBED_MODEL = "nomic-embed-text"  #@param {type:"string"}
#@markdown Số luồng gọi Ollama song song (1 = tuần tự, tận dụng KV cache tiền tố tốt nhất).
WORKERS = 1  #@param {type:"integer"}
#@markdown Lưu kết quả vào Google Drive (khuyến nghị).
LUU_DRIVE = True  #@param {type:"boolean"}

MODELS = [m for m, on in [("qwen3.5:0.8b", CHAY_qwen35_08b), ("qwen3:8b", CHAY_qwen3_8b), ("qwen3.5:9b", CHAY_qwen35_9b)] if on]
METHODS = "all" if ABLATION else "main"
REPO = "/content/vsqlplus"
DATA = "/content/data/minidev/MINIDEV"
DRIVE_ROOT = "/content/drive/MyDrive/vsqlplus_ketqua"
TAG = "" if not SAMPLE else f"_mau{SAMPLE}"
OUT = {m: f"/content/ketqua/cmp_{m.replace(':', '_').replace('.', '')}{TAG}" for m in MODELS}
print("Model:", MODELS, "| cấu hình:", METHODS, "| số câu:", SAMPLE or 500)
assert MODELS, "Chọn ít nhất một model"

In [ ]:
#@title Bước 2. Kiểm tra GPU, kết nối Google Drive
import subprocess, re, os
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
CO_GPU = bool(gpu.stdout.strip())
print("GPU:", gpu.stdout.strip() or "KHÔNG CÓ GPU")
if not CO_GPU:
    print("CẢNH BÁO: chưa bật GPU (Runtime -> Change runtime type -> T4 GPU, rồi chạy lại từ Bước 1). "
          "Không có GPU chỉ chạy được phần xem/gộp kết quả có sẵn (Bước 7, 10, 11, 12).")
if LUU_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_ROOT, exist_ok=True)

In [ ]:
#@title Bước 3. Tải mã nguồn và chạy kiểm thử (không cần GPU, ~1 phút)
import os
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q --depth 1 https://github.com/chloenguyen167/vsqlplus.git {REPO}
%cd {REPO}
!python tests/test_compare.py | tail -1
!python tests/test_all.py | tail -1

In [ ]:
#@title Bước 4. Tải BIRD mini-dev (~800 MB) và mô tả cột của TA-SQL
import os, shutil, json, urllib.request
ZIP_DRIVE = f"{DRIVE_ROOT}/minidev.zip"
if not os.path.exists(f"{DATA}/mini_dev_sqlite.json"):
    os.makedirs("/content/data", exist_ok=True)
    if LUU_DRIVE and os.path.exists(ZIP_DRIVE):
        shutil.copy(ZIP_DRIVE, "/content/data/minidev.zip")
    else:
        !wget -q --show-progress -O /content/data/minidev.zip https://bird-bench.oss-cn-beijing.aliyuncs.com/minidev.zip
        if LUU_DRIVE:
            shutil.copy("/content/data/minidev.zip", ZIP_DRIVE)
    !unzip -q -o /content/data/minidev.zip "minidev/MINIDEV/*" -d /content/data
d = json.load(open(f"{DATA}/mini_dev_sqlite.json"))
print(len(d), "dòng dữ liệu (tính đủ, kể cả question_id 137/138 xuất hiện hai lần như bộ đánh giá chính thức);",
      len(os.listdir(f"{DATA}/dev_databases")), "CSDL")
# column_meaning.json: mô tả cột do TA-SQL sinh sẵn (TASL dùng trong prompt dummy SQL)
MEANING = "/content/data/tasql_column_meaning.json"
if not os.path.exists(MEANING):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/quge2023/TA-SQL/master/outputs/column_meaning.json", MEANING)
print("column_meaning.json:", len(json.load(open(MEANING))), "cột")

In [ ]:
#@title Bước 5. Sinh view và đo độ phủ bước chọn view (không gọi LLM, ~2 phút)
CHUNG = (f"--data {DATA}/mini_dev_sqlite.json --db-root {DATA}/dev_databases --tables-json {DATA}/dev_tables.json "
         f"--tasql-meaning {MEANING}")
!python run_compare.py {CHUNG} --out /content/ketqua/_views --views-only
print("\nĐộ phủ bảng đáp án của bước chọn view V-SQL+ (BM25 + liên kết giá trị, chưa embedding / bầu chọn):")
!python run_compare.py {CHUNG} --out /content/ketqua/_views --selection-sweep --no-embed | tail -9

In [ ]:
#@title Bước 6. Cài Ollama, tải model và khởi động (~5–10 phút)
import subprocess, time, urllib.request, json, os
if not CO_GPU:
    raise SystemExit("Bước này cần GPU: Runtime -> Change runtime type -> T4 GPU, rồi chạy lại từ Bước 1.")
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama_install.log 2>&1; tail -1 /tmp/ollama_install.log
env = dict(os.environ, OLLAMA_FLASH_ATTENTION="1", OLLAMA_KV_CACHE_TYPE="q8_0", OLLAMA_NUM_PARALLEL=str(max(1, WORKERS)),
           OLLAMA_MAX_LOADED_MODELS="2", OLLAMA_KEEP_ALIVE="-1", OLLAMA_HOST="127.0.0.1:11434")
if not globals().get("ollama_proc") or ollama_proc.poll() is not None:
    ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2); break
    except Exception:
        time.sleep(1)
print("Ollama:", urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode())
for m in MODELS + ([EMBED_MODEL] if EMBED_MODEL else []):
    print("Tải", m)
    !ollama pull {m} 2>&1 | tail -1

def thu(model):
    body = {"model": model, "stream": False, "think": False, "options": {"temperature": 0, "num_ctx": 20480, "num_predict": 60},
            "messages": [{"role": "user", "content": "Write one SQLite query that counts rows of table t. Return only SQL."}]}
    t = time.time()
    r = json.loads(urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:11434/api/chat",
        json.dumps(body).encode(), {"Content-Type": "application/json"}), timeout=900).read())
    return time.time() - t, r["message"]["content"][:80].replace("\n", " "), r.get("eval_count", 0) / max(1e-9, r.get("eval_duration", 1) / 1e9)
for m in MODELS:
    thu(m)                                  # lần đầu nạp model vào GPU
    dt, txt, tps = thu(m)
    print(f"{m}: {dt:.1f}s, ~{tps:.0f} token/s -> {txt}")
if EMBED_MODEL:
    r = json.loads(urllib.request.urlopen(urllib.request.Request("http://127.0.0.1:11434/api/embed",
        json.dumps({"model": EMBED_MODEL, "input": ["probe"]}).encode(), {"Content-Type": "application/json"})).read())
    print("Embedding", EMBED_MODEL, "chiều:", len(r["embeddings"][0]))
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
#@title Bước 7. Đồng bộ Google Drive (khôi phục phần đã chạy + tự sao lưu mỗi 5 phút)
import os, shutil, sqlite3, threading, time

def sao_luu():
    if not LUU_DRIVE:
        return
    for m, src in OUT.items():
        if not os.path.isdir(src):
            continue
        dst = f"{DRIVE_ROOT}/{os.path.basename(src)}"
        for root, _, files in os.walk(src):
            rel = os.path.relpath(root, src)
            os.makedirs(os.path.join(dst, rel), exist_ok=True)
            for f in files:
                a, b = os.path.join(root, f), os.path.join(dst, rel, f)
                if f.endswith(".sqlite"):                 # cache LLM: chép bằng backup API, không hỏng khi đang ghi
                    with sqlite3.connect(a) as s, sqlite3.connect(b + ".tmp") as d:
                        s.backup(d)
                    os.replace(b + ".tmp", b)
                elif not os.path.exists(b) or os.path.getmtime(a) > os.path.getmtime(b) + 1:
                    shutil.copy2(a, b)

for m, src in OUT.items():
    dst = f"{DRIVE_ROOT}/{os.path.basename(src)}"
    if LUU_DRIVE and os.path.isdir(dst) and not os.path.isdir(src):
        shutil.copytree(dst, src)
        print("Khôi phục", m, "từ Drive:", sorted(os.listdir(src))[:10])

def _vong():
    while True:
        time.sleep(300)
        try:
            sao_luu()
        except Exception as e:
            print("[sao lưu] lỗi:", e)

if LUU_DRIVE and not globals().get("_dang_sao_luu"):
    threading.Thread(target=_vong, daemon=True).start()
    _dang_sao_luu = True
    print("Tự sao lưu mỗi 5 phút ->", DRIVE_ROOT)

In [ ]:
#@title Bước 8. Chạy thử 3 câu × 3 phương pháp với model đầu tiên (~2–5 phút)
EMB = f"--embed-model {EMBED_MODEL}" if EMBED_MODEL else "--no-embed"
!rm -rf /content/ketqua/_thu
!python run_compare.py {CHUNG} --model {MODELS[0]} {EMB} --out /content/ketqua/_thu --limit 3 --methods {METHODS} 2>&1 | grep -E "^\| \`|Error|Trace|CANH BAO" | head -12

## Bước 9. Chạy chính
Mỗi model chạy 3 phương pháp (`tasql`, `vsql`, `vsqlplus`; thêm 5 biến thể của V-SQL+ nếu bật ABLATION). Chương trình gom các
prompt còn thiếu theo từng vòng và sắp theo thứ tự chữ để Ollama dùng lại KV cache; dòng `[prefetch vong k] ... con lai ~N phut`
cho biết thời gian còn lại của vòng đó (TA-SQL 3 vòng, V-SQL 2, V-SQL+ tối đa 4).

**Nếu Colab bị ngắt:** mở lại notebook, chạy Bước 1 → 7, rồi chạy lại ô này.

In [ ]:
#@title Bước 9. Chạy chính (lâu nhất)
import subprocess
if not CO_GPU:
    raise SystemExit("Bước này cần GPU: Runtime -> Change runtime type -> T4 GPU, rồi chạy lại từ Bước 1.")
MAU = f"--sample {SAMPLE}" if SAMPLE else ""
for m in MODELS:
    cmd = (f"python -u run_compare.py {CHUNG} --model {m} {EMB} --workers {WORKERS} --methods {METHODS} "
           f"--out {OUT[m]} {MAU}")
    print("\n=====", m, "=====\n" + cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if not line.startswith("[schema]"):
            print(line, end="")
    p.wait()
    sao_luu()
    print(m, "xong, mã thoát:", p.returncode)

In [ ]:
#@title Bước 10. Bảng kết quả theo từng model, kiểm định ghép cặp và so với paper
import csv, os, sys
from IPython.display import Markdown, display
sys.path.insert(0, REPO)
from vsql import report
TEN = {"tasql": "TA-SQL", "vsql": "V-SQL (paper)", "vsqlplus": "V-SQL+ (cải tiến)",
       "vsqlplus_all": "Chỉ hướng 1 (view tự động, không chọn view)", "vsqlplus_novote": "V-SQL+ không bầu chọn",
       "vsqlplus_novl": "V-SQL+ không liên kết giá trị", "vsqlplus_nofb": "V-SQL+ không schema lai",
       "vsqlplus_det": "V-SQL+ giải mã tất định"}
DIFF = ["simple", "moderate", "challenging"]
tong = {}
for m, out in OUT.items():
    if not os.path.exists(out) or not any(f.startswith("predictions_") for f in os.listdir(out)):
        print("Chưa có kết quả cho", m); continue
    report.build(out, refs=["tasql", "vsql"])
    S = {r["method"]: r for r in csv.DictReader(open(f"{out}/summary.csv"))}
    tong[m] = S
    rows = ["| Cấu hình | n | SIM. | MOD. | CHALL. | **TOTAL** | CI 95% | Ảo giác schema (%) | Gọi LLM/câu | Token vào/câu |",
            "|---|---|---|---|---|---|---|---|---|---|"]
    for k in TEN:
        if k in S:
            x = S[k]
            rows.append(f"| {TEN[k]} | {x['n']} | " + " | ".join(f"{float(x['ex_' + d]):.1f}" if x.get('ex_' + d) else "-" for d in DIFF)
                        + f" | **{float(x['ex']):.1f}** | {float(x['ci_lo']):.1f}–{float(x['ci_hi']):.1f} | "
                        f"{float(x['schema_hallucination']):.1f} | {float(x['calls']):.2f} | {float(x['prompt_tokens']):.0f} |")
    cmp = ["| So sánh | ΔEX (điểm) | CI 95% | chỉ A đúng / chỉ B đúng | p McNemar | p Holm |", "|---|---|---|---|---|---|"]
    for a, b, sfx in [("vsqlplus", "tasql", ""), ("vsqlplus", "vsql", "_vsql"), ("vsql", "tasql", ""),
                      ("vsqlplus_all", "vsql", "_vsql")]:
        if a in S and S[a].get("delta_vs_ref" + sfx):
            x = S[a]
            cmp.append(f"| {TEN[a]} − {TEN[b]} | {float(x['delta_vs_ref' + sfx]):+.1f} | "
                       f"[{float(x['delta_lo' + sfx]):+.1f}, {float(x['delta_hi' + sfx]):+.1f}] | "
                       f"{x['only_this' + sfx]} / {x['only_ref' + sfx]} | {float(x['mcnemar_p' + sfx]):.3f} | {float(x['holm_p' + sfx]):.3f} |")
    display(Markdown(f"### {m}\n" + "\n".join(rows) + "\n\n" + "\n".join(cmp)))

display(Markdown("""### Tham chiếu: Bảng 3 của paper V-SQL (EX %, BIRD mini-dev, cùng 500 câu)
| Model | Phương pháp | SIM. | MOD. | CHALL. | TOTAL |
|---|---|---|---|---|---|
| GPT-4 | TA-SQL | 62.83 | 60.4 | 36.11 | 56.17 |
| GPT-4 | V-SQL | 62.84 | 59.6 | 37.25 | 56.0 |
| GPT-4-turbo | TA-SQL | 62.9 | 48.4 | 36.11 | 50.16 |
| GPT-4-turbo | V-SQL | 62.16 | 51.6 | 40.19 | 52.8 |
| GPT-3.5-turbo | TA-SQL | 56.75 | 42.8 | 27.45 | 43.8 |
| GPT-3.5-turbo | V-SQL | 58.11 | 43.6 | 31.37 | 45.4 |

Số tuyệt đối không so trực tiếp được (khác model); điều so được là **thứ tự và chênh lệch giữa các phương pháp trên cùng model**."""))

In [ ]:
#@title Bước 11. Tổng hợp giữa các model + biểu đồ
import matplotlib.pyplot as plt
if tong:
    keys = [k for k in ["tasql", "vsql", "vsqlplus"] if any(k in S for S in tong.values())]
    rows = ["| Model | " + " | ".join(TEN[k] for k in keys) + " |", "|---|" + "---|" * len(keys)]
    for m, S in tong.items():
        rows.append(f"| {m} | " + " | ".join(f"{float(S[k]['ex']):.1f}" if k in S else "-" for k in keys) + " |")
    display(Markdown("### EX (%) theo model\n" + "\n".join(rows)))
    fig, ax = plt.subplots(figsize=(8, 3.6))
    w = 0.8 / len(keys)
    mau = ["#2a78d6", "#e8833a", "#1baf7a"]
    for i, k in enumerate(keys):
        xs = [j + i * w for j in range(len(tong))]
        ys = [float(S[k]["ex"]) if k in S else 0 for S in tong.values()]
        lo = [float(S[k]["ex"]) - float(S[k]["ci_lo"]) if k in S else 0 for S in tong.values()]
        hi = [float(S[k]["ci_hi"]) - float(S[k]["ex"]) if k in S else 0 for S in tong.values()]
        ax.bar(xs, ys, w, yerr=[lo, hi], capsize=2, label=TEN[k], color=mau[i % len(mau)])
    ax.set_xticks([j + w * (len(keys) - 1) / 2 for j in range(len(tong))], list(tong))
    ax.set_ylabel("EX (%)"); ax.legend(frameon=False, fontsize=8, ncol=2); ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.savefig("/content/ketqua/bieu_do_ex.png", dpi=150); plt.show()

In [ ]:
#@title Bước 12. Nén và tải kết quả về máy
import shutil, os
from google.colab import files
sao_luu()
z = shutil.make_archive("/content/ketqua_tasql_vsql_vsqlplus", "zip", "/content/ketqua")
print("Kích thước:", round(os.path.getsize(z) / 1e6, 1), "MB")
files.download(z)

## Gặp lỗi?
- **Ollama chạy rất chậm / `CUDA out of memory`**: tắt bớt model khác (mỗi phiên một model), giữ `WORKERS = 1`.
- **Colab ngắt giữa chừng**: chạy lại Bước 1 → 7 rồi Bước 9; phần đã xong khôi phục từ Drive (cache LLM nằm trong `llm_cache.sqlite`).
- **`[canh bao] khong goi duoc embedding`**: V-SQL+ tự chuyển sang chỉ BM25; kiểm tra `ollama pull nomic-embed-text`.
- **`prompt cham nguong num_ctx`**: tăng `--num-ctx` (mặc định 20480).